# Counterfactuals — Pendulum

DDIM inversion + per-attribute intervention sampling for the synthetic
pendulum benchmark (4 causal attributes: pendulum angle, light angle,
shadow length, shadow position). Pipeline assembly is shared across the
three benchmark notebooks via `inference_utils.py`; this notebook only
owns the dataset loader and the intervention sweep.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
# Two extra entries on sys.path:
#   - the package root, so `causal_modules` / `causal_datasets` resolve.
#   - this notebook's own folder, so `inference_utils` resolves regardless
#     of the kernel's current working directory.
sys.path.append(str(Path.cwd().resolve().parent))  # repo root
sys.path.append(str(Path.cwd().resolve()))  # this notebook's folder

import os
import random

import numpy as np
import torch
from PIL import Image
from torchvision import transforms

from causal_modules.ddim_modules import sample, ddim_editing, save_images_grid
from inference_utils import build_transforms, load_causal_adapter

## 1. Configuration

All paths are derived from two roots:

* `MODEL_CACHE` — local HuggingFace snapshot cache holding the frozen
  Stable Diffusion v1.5 backbone (here the **miniSD** variant).
* `LOGS_ROOT` — directory holding the training-run sub-folders produced by
  `train.py` (the Causal-Adapter ControlNet + MCPL embeddings) and the
  separate SCM pretraining runs from `SCM_modeling/`.

Each downstream variable picks one specific run / step from those roots.

In [ ]:
# Shared roots
MODEL_CACHE = os.environ.get('MODEL_CACHE', '')
LOGS_ROOT = os.environ.get('LOGS_ROOT', '')

# 1) Frozen SD1.5 backbone (`lambdalabs/miniSD-diffusers`). A HF Hub id
#    works equally well — the local snapshot is just faster behind firewalls.
BASE_MODEL_PATH = ''

# 2) Causal-Adapter ControlNet + MCPL learned pseudo-tokens. Both files come
#    from the SAME training run — mismatching them silently breaks inference.
CONTROLNET_PATH     = ''
TEXT_EMBEDDING_PATH = ''

# 3) (Optional) Pretrained SCM head from `SCM_modeling/`. Setting this to
#    None keeps the head weights stored inside CONTROLNET_PATH untouched.
SCM_PATH = ''

# 4) Folder containing pendulum images named `a_<pendulum>_<light>_<shadow_l>_<shadow_p>.png`.
DATA_ROOT = os.environ.get('DATA_ROOT', '')

DATASET = 'pendulum'
SIZE = 256

## 2. Build pipeline + transforms

`load_causal_adapter` does five things in one call: load the
`Causal_ControlNetModel`, optionally overwrite its SCM head with the
pretraining-stage weights, apply the dataset's adjacency mask, load MCPL
pseudo-token embeddings into a fresh `CLIPTextModel`, and assemble the
`StableDiffusionCausalControlNetPipeline` with the DDIM scheduler config
required for inversion (`clip_sample=False`, `set_alpha_to_one=False`).

In [ ]:
image_transforms, original_transforms, _ = build_transforms(DATASET, size=SIZE)

assets = load_causal_adapter(
    DATASET,
    base_model_path=BASE_MODEL_PATH,
    controlnet_path=CONTROLNET_PATH,
    text_embedding_path=TEXT_EMBEDDING_PATH,
    scm_path=SCM_PATH,
)
pipe              = assets.pipe                # StableDiffusionCausalControlNetPipeline
device            = assets.device              # torch.device('cuda') by default
prompt            = assets.prompt              # default training prompt for this dataset
presudo_words     = assets.presudo_words       # comma-joined MCPL pseudo-tokens
presudo_list      = assets.presudo_list        # ['@', '*', '&', '!']
presudo_token_ids = assets.presudo_token_ids   # tokenizer ids of the pseudo-tokens

## 3. Load a pendulum sample

Pendulum filenames embed the four ground-truth causal attributes; we
invert the same per-attribute Gaussian normalisation used during training
so the SCM head sees inputs on the right scale.

In [ ]:
def normalize_label_gaussian(label):
    # Per-attribute (mean, spread) used at training time:
    #   pendulum angle, light angle, shadow length, shadow position.
    scale = np.array([[2, 42], [104, 44], [7.5, 4.5], [11, 8]])
    return ((label - scale[:, 0]) / scale[:, 1]).astype(np.float32)


def load_pendulum_sample(data_root=DATA_ROOT, img_path=None, device=device):
    if img_path is None:
        files = [f for f in os.listdir(data_root) if f.endswith('.png')]
        img_path = os.path.join(data_root, random.choice(files))

    label_values = list(map(float, os.path.basename(img_path)[:-4].split('_')[1:]))
    norm_label = normalize_label_gaussian(np.array(label_values))
    image = Image.open(img_path)

    print(f'Loaded image: {img_path}')
    print(f'Raw label: {label_values}')
    print(f'Normalized label: {norm_label}')
    return image, torch.tensor(norm_label).unsqueeze(0).to(device).float()


img, label = load_pendulum_sample()
img

## 4. DDIM inversion + reconstruction

`ddim_editing` does two passes: first inverts the source image to a noisy
latent (so we can re-decode it later under modified conditioning), then
samples back to a reconstruction. With no intervention this should yield
a near-identical image — it is our "identity" sanity check.

In [ ]:
image = img.convert('RGB') if img.mode != 'RGB' else img
original_img = original_transforms(image.copy())
condition_image = image.copy()
image_t = image_transforms(image)

set_guidance_scale = 1.0   # CFG scale during the forward pass; 1.0 = no CFG.
num_steps = 50             # DDIM steps for inversion *and* sampling.

final_im, inverted_latents, _, uncond_embeddings = ddim_editing(
    pipe,
    image_t.unsqueeze(0),
    label.clone(),
    presudo_token_ids,
    prompt,
    num_steps=num_steps,
    invert_guidance_scale=1.0,
    set_guidance_scale=set_guidance_scale,
    intervention_indx=None,        # no intervention
    intervention_values=None,
    return_PIL=True,
)
save_images_grid([[original_img, final_im[0]]], (1, 2), None)

## 5. Per-attribute intervention sweep

For each of the four pendulum attributes (`intervention_indx` 0..3) we
linearly sweep `intervention_values` across a sensible range and run the
sampler from the inverted latent. The sweep ranges below match what the
Gaussian-normalised SCM head saw during training.

In [ ]:
import matplotlib.pyplot as plt


def save_images_grid_box(images_list, grid_size, save_path=None, border=1):
    """Tiled grid with a thin black border around each cell."""
    images = [np.asarray(img) for sublist in images_list for img in sublist]
    H, W, C = images[0].shape
    grid_rows, grid_cols = grid_size
    assert grid_rows * grid_cols >= len(images), 'Grid size too small'

    canvas = np.zeros(
        (grid_rows * (H + border) + border, grid_cols * (W + border) + border, C),
        dtype=np.uint8,
    )
    for idx, im in enumerate(images):
        r, c = divmod(idx, grid_cols)
        rs = r * (H + border) + border
        cs = c * (W + border) + border
        canvas[rs : rs + H, cs : cs + W] = im

    plt.figure(figsize=(grid_cols * 2, grid_rows * 2))
    plt.imshow(canvas)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    if save_path is not None:
        plt.savefig(save_path, bbox_inches='tight')

In [ ]:
range_len = 8
# Per-attribute target ranges (in normalised space).
# Index meaning: 0 pendulum, 1 light, 2 shadow length, 3 shadow position.
inter_value = [
    np.linspace(-1.0, 1.0, num=range_len),
    np.linspace(-0.5, 0.5, num=range_len),
    np.linspace(-0.5, 0.5, num=range_len),
    np.linspace(-0.5, 0.5, num=range_len),
]

image_lists = []
for inter_id in range(4):
    images = []
    for i in range(range_len):
        # `intervention_indx` selects which causal node to override.
        # `intervention_values` is the value we set that node to (in normalised space).
        # The SCM head propagates the override to the children so downstream
        # attributes stay causally consistent.
        interved_image, _ = sample(
            pipe,
            prompt,
            presudo_token_ids,
            start_step=0,
            start_latents=inverted_latents[-1].clone(),
            guidance_scale=1.0,
            num_inference_steps=50,
            num_images_per_prompt=1,
            negative_prompt=None,
            device=device,
            controlnet_image=None,
            intervention_indx=inter_id,
            intervention_values=inter_value[inter_id][i],
            label=label.clone(),
            return_PIL=True,
            disentangle=False,
            uncond_embeddings=uncond_embeddings,
        )
        images.append(interved_image[0])
    image_lists.append([np.asarray(im) for im in images])

save_images_grid_box(image_lists, (4, range_len), './intervention_pendulum.png')

## 6. Cross-attention maps

Visualises which spatial regions each MCPL pseudo-token attends to during
denoising. A clean run shows each pseudo-token bound to one visual
attribute; a noisy run hints at concept entanglement.

In [ ]:
from causal_modules.p2p_edits import ptp_tools, ptp_utils
import importlib
importlib.reload(ptp_tools)
importlib.reload(ptp_utils)

out_dir = os.path.join('./textcond', prompt)
os.makedirs(out_dir, exist_ok=True)

generator = torch.manual_seed(0)
overlapped_mask, attn_img = ptp_tools.plot_img_attn_mask_textcontrol(
    pipe, [prompt], presudo_words, presudo_token_ids, condition_image,
    device, out_dir, 'causalnet.png',
    latent=image_t.unsqueeze(0), res=16, label=label,
    GUIDANCE_SCALE=2.0, attn_threshold=0.5, only_sampling=False,
    dataset='pendulum', show_text=True, save_masks=False, class_select=False,
    intervention_indx=None, intervention_values=None,
    from_where=['down', 'up'], mask_concepts=True,
    g_gpu=generator, num_steps=30, img_size=SIZE, exp_names=['textcond'],
)